In [3]:
import cv2
import numpy as np
import time
from tensorflow.keras.models import load_model
from playsound import playsound
import threading

In [2]:
model = load_model("eye_state_model.keras")


In [3]:
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')


In [28]:
'''import cv2

# Load image
img_path = "test2.jpg"
img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Load eye detector
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

# Detect eyes
eyes = eye_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

for (x, y, w, h) in eyes[:1]:  # Take only the first detected eye
    eye_roi = gray[y:y+h, x:x+w]
    eye_resized = cv2.resize(eye_roi, (32, 32))
    eye_normalized = eye_resized / 255.0
    eye_input = eye_normalized.reshape(1, 32, 32, 1)

    pred = model.predict(eye_input)[0][0]
    print("Prediction Score:", pred)
    if pred > 0.5:
        print("Predicted: Open Eye")
    else:
        print("Predicted: Closed Eye")
'''

1/1 [==============================] - 0s 17ms/step
Prediction Score: 0.0029569692
Predicted: Closed Eye


In [8]:
closed_frames = 0
threshold_frames = 30
alarm_playing = False

def play_alarm():
    global alarm_playing
    playsound("alarm.mp3")  
    alarm_playing = False
    

In [9]:
cap=cv2.VideoCapture(0)
while True:
    ret,frame=cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    eyes = eye_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    eye_closed_in_frame = False

    for (x, y, w, h) in eyes[:1]:  
        eye_roi = gray[y:y+h, x:x+w]
        eye_resized = cv2.resize(eye_roi, (32, 32))
        eye_normalized = eye_resized / 255.0
        eye_input = eye_normalized.reshape(1, 32, 32, 1)
        pred = model.predict(eye_input, verbose=0)[0][0]    
    
    label = "Open" if pred > 0.5 else "Closed"
    color = (0, 255, 0) if pred > 0.5 else (0, 0, 255)
    cv2.putText(frame, f"Eye: {label} ({pred:.2f})", (x, y - 10),
        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

    if pred <= 0.5:
        eye_closed_in_frame = True
        # Update counter
    if eye_closed_in_frame:
        closed_frames += 1
    else:
        closed_frames = 0
        alarm_playing = False

    # Trigger alarm if threshold is crossed
    if closed_frames >= threshold_frames and not alarm_playing:
        alarm_playing = True
        threading.Thread(target=play_alarm).start()

    # Show video frame
    cv2.imshow("Eye State Monitor", frame)

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
cv2.destroyAllWindows()    

In [10]:


# Load cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

# Load model

def play_alarm():
    global alarm_playing
    playsound("alarm.mp3")  
    alarm_playing = False
    

# Parameters
threshold_frames = 20
closed_frames = 0
alarm_playing = False

# Video capture
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

    eye_closed_in_frame = False

    for (fx, fy, fw, fh) in faces:
        face_roi_gray = gray[fy:fy+fh, fx:fx+fw]
        face_roi_color = frame[fy:fy+fh, fx:fx+fw]

        eyes = eye_cascade.detectMultiScale(face_roi_gray, scaleFactor=1.1, minNeighbors=5)

        for (ex, ey, ew, eh) in eyes[:1]:  # Use only one eye for prediction
            eye_gray = face_roi_gray[ey:ey+eh, ex:ex+ew]
            eye_resized = cv2.resize(eye_gray, (32, 32))
            eye_normalized = eye_resized / 255.0
            eye_input = eye_normalized.reshape(1, 32, 32, 1)

            pred = model.predict(eye_input, verbose=0)[0][0]
            label = "Open" if pred > 0.5 else "Closed"
            color = (0, 255, 0) if pred > 0.5 else (0, 0, 255)

            # Draw results relative to the full frame
            cv2.putText(frame, f"Eye: {label} ({pred:.2f})",
                        (fx + ex, fy + ey - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.rectangle(frame, (fx + ex, fy + ey), (fx + ex + ew, fy + ey + eh), color, 2)

            if pred <= 0.5:
                eye_closed_in_frame = True

        break  # Only process one face for now

    # Update counter
    if eye_closed_in_frame:
        closed_frames += 1
    else:
        closed_frames = 0
        alarm_playing = False

    # Trigger alarm
    if closed_frames >= threshold_frames and not alarm_playing:
        alarm_playing = True
        threading.Thread(target=play_alarm).start()

    # Show video frame
    cv2.imshow("Eye State Monitor", frame)

    # Exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import threading
from keras.models import load_model
import

# Load model
model = load_model("eye_state_model.keras")

# Alarm function
def play_alarm():
    global alarm_playing
    playsound("alarm.mp3")  
    alarm_playing = False

# Mediapipe face mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1)

# Eye indices (right eye example)
RIGHT_EYE_IDX = [33, 133, 160, 159, 158, 144, 153, 154, 155, 173]

threshold_frames = 20
closed_frames = 0
alarm_playing = False

In [5]:


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(rgb)

    eye_closed_in_frame = False

    if result.multi_face_landmarks:
        for face_landmarks in result.multi_face_landmarks:
            h, w, _ = frame.shape
            right_eye_pts = [(int(landmark.x * w), int(landmark.y * h)) for i, landmark in enumerate(face_landmarks.landmark) if i in RIGHT_EYE_IDX]

            # Get bounding box around right eye
            x_coords = [pt[0] for pt in right_eye_pts]
            y_coords = [pt[1] for pt in right_eye_pts]
            x_min, x_max = min(x_coords), max(x_coords)
            y_min, y_max = min(y_coords), max(y_coords)

            # Add padding
            pad = 5
            x_min = max(0, x_min - pad)
            y_min = max(0, y_min - pad)
            x_max = min(w, x_max + pad)
            y_max = min(h, y_max + pad)

            # Crop eye region
            eye_img = frame[y_min:y_max, x_min:x_max]
            gray_eye = cv2.cvtColor(eye_img, cv2.COLOR_BGR2GRAY)
            eye_resized = cv2.resize(gray_eye, (32, 32))
            eye_normalized = eye_resized / 255.0
            eye_input = eye_normalized.reshape(1, 32, 32, 1)

            pred = model.predict(eye_input, verbose=0)[0][0]
            label = "Open" if pred > 0.5 else "Closed"
            color = (0, 255, 0) if pred > 0.5 else (0, 0, 255)

            cv2.putText(frame, f"Eye: {label} ({pred:.2f})", (x_min, y_min - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 2)

            if pred <= 0.5:
                eye_closed_in_frame = True
            break

    if eye_closed_in_frame:
        closed_frames += 1
    else:
        closed_frames = 0
        alarm_playing = False

    if closed_frames >= threshold_frames and not alarm_playing:
        alarm_playing = True
        threading.Thread(target=play_alarm).start()

    cv2.imshow("Eye State Monitor", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



    Error 263 for command:
        close alarm.mp3
    The specified device is not open or is not recognized by MCI.
Failed to close the file: alarm.mp3
